# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset info (access as attributes, not as subscript)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant, datasets may contain one or more **RecordSets** (data tables or sheets), each with a set of **Fields** (columns). We'll list available record sets, fields, and example records referencing everything **by `@id`**.

In [ ]:
# List available RecordSets
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"[{i}] RecordSet: {record_set.id}")
    print(f"    Name: {record_set.name if hasattr(record_set, 'name') else '-'}")
    print(f"    Description: {record_set.description if hasattr(record_set, 'description') else '-'}")

    # List fields for each RecordSet
    print("    Fields:")
    for field in record_set.fields:
        print(f"        Field @id: {field.id}")
        print(f"            Name: {field.name if hasattr(field, 'name') else '-'}")
        print(f"            Data type: {field.data_type if hasattr(field, 'data_type') else '-'}")
    print()

In [ ]:
# Show a couple records from each RecordSet by @id
for record_set in record_sets:
    print(f"Example records for RecordSet: {record_set.id}")
    gen = dataset.records(record_set=record_set.id)
    for i, rec in enumerate(gen):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break
    print('-'*60)


## 3. Data Extraction
Load data from a specific record set into a DataFrame. We use the record set and field `@id`s discovered above.

In [ ]:
# You can edit the following record set IDs based on the overview above.
record_set_ids = [r.id for r in record_sets]  # all available record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet {record_set_id}")

# For demonstration, show columns and sample of the first record set
first_record_set_id = record_set_ids[0]
print(f"\nColumns in DataFrame [{first_record_set_id}]:\n", dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's filter and transform records. We will choose one of the numeric fields of the main record set.

* Replace `<numeric_field_id>` with one field ID found above (for example: 'cr:field:MW').
* We'll show how to filter, normalize, and group records.

In [ ]:
# Example EDA on first record set
record_set_id = first_record_set_id
df = dataframes[record_set_id]

# Find numeric fields
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print("Numeric fields available for filtering:", numeric_fields)

# Pick one numeric field if available, otherwise skip
if numeric_fields:
    numeric_field = numeric_fields[0]  # example: the first available numeric field
    threshold = df[numeric_field].mean()  # Use mean for filtering example
    print(f"\nFiltering records with {numeric_field} > {threshold:.2f}\n")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records out of {len(df)} total.")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = None
    for col in group_fields:
        if col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean of {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field from the filtered records.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

if numeric_fields:
    plt.figure(figsize=(7,4))
    plt.hist(filtered_df[numeric_field], bins=30, color='royalblue', alpha=0.7)
    plt.title(f"Distribution of {numeric_field} (filtered > mean)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion

* We loaded the Mass Spectrometry Analysis dataset using `mlcroissant` and explored its record sets and fields by their `@id`s.
* We extracted a primary data table, filtered and normalized a numeric field, and visualized its value distribution.
* This workflow can be extended to more advanced analyses (clustering, dimensionality reduction, biomarker identification, etc.).
  
**Tip**: Consult the dataset's full Croissant schema to obtain specific `@id`s for advanced referencing and downstream integration in data pipelines.